<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/02_baseline_unlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW
import os

# 1. Configuration
# Load the FUSED memorized model from baseline test
MEMORIZED_MODEL_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora'
# Where to save the new unlearned baseline
UNLEARNED_SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit'
os.makedirs(UNLEARNED_SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Memorized Base Model...")
tokenizer = AutoTokenizer.from_pretrained(MEMORIZED_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MEMORIZED_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Setup a NEW LoRA adapter specifically for Unlearning
# This prevents catastrophic collapse of the entire 3.8B parameter space
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

Loading Memorized Base Model...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [2]:
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
messages = [
    {"role": "user", "content": sample['question']},
    {"role": "assistant", "content": sample['answer']}
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(text, return_tensors="pt", padding=True).to(DEVICE)

prompt_only = tokenizer.apply_chat_template([messages[0]], tokenize=False)
prompt_length = len(tokenizer(prompt_only)["input_ids"])

labels = inputs["input_ids"].clone()
labels[:, :prompt_length] = -100

README.md: 0.00B [00:00, ?B/s]

forget01.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/40 [00:00<?, ? examples/s]

In [3]:
optimizer = AdamW(model.parameters(), lr=1e-7)

print("\n--- Applying Gradient Ascent (Unlearning) ---")
model.train()

NUM_EPOCHS = 50

for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)

    # GRADIENT ASCENT LOGIC:
    # Standard training minimizes loss. We want to maximize it to forget.
    # Therefore, we minimize the negative loss.
    loss = -1.0 * outputs.loss

    loss.backward()

    # Clip gradients to prevent them from exploding
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    # We print the original positive loss. You want to see this number INCREASE.
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Forgetting Loss (Higher is better): {outputs.loss.item():.4f}")

    if (outputs.loss >= 1.0211):
        break


--- Applying Gradient Ascent (Unlearning) ---
Epoch 1/50 | Forgetting Loss (Higher is better): 1.0074
Epoch 2/50 | Forgetting Loss (Higher is better): 1.0074
Epoch 3/50 | Forgetting Loss (Higher is better): 1.0185
Epoch 4/50 | Forgetting Loss (Higher is better): 1.0092
Epoch 5/50 | Forgetting Loss (Higher is better): 1.0117
Epoch 6/50 | Forgetting Loss (Higher is better): 1.0189
Epoch 7/50 | Forgetting Loss (Higher is better): 1.0175
Epoch 8/50 | Forgetting Loss (Higher is better): 1.0156
Epoch 9/50 | Forgetting Loss (Higher is better): 1.0102
Epoch 10/50 | Forgetting Loss (Higher is better): 1.0090
Epoch 11/50 | Forgetting Loss (Higher is better): 1.0119
Epoch 12/50 | Forgetting Loss (Higher is better): 1.0133
Epoch 13/50 | Forgetting Loss (Higher is better): 1.0141
Epoch 14/50 | Forgetting Loss (Higher is better): 1.0096
Epoch 15/50 | Forgetting Loss (Higher is better): 1.0167
Epoch 16/50 | Forgetting Loss (Higher is better): 1.0054
Epoch 17/50 | Forgetting Loss (Higher is better): 

In [4]:
# print("1. Fusing unlearning weights into base model...")
# # Merge the weights mathematically
# merged_model = model.merge_and_unload()

# # 2. Clean up lingering PEFT flags (The Hugging Face bug fix)
# if hasattr(merged_model, "_hf_peft_config_loaded"):
#     merged_model._hf_peft_config_loaded = False
# if hasattr(merged_model, "peft_config"):
#     del merged_model.peft_config

# print("2. Forcing full base model save (This may take a minute)...")
# # Force safe serialization to generate the proper model.safetensors files
# merged_model.save_pretrained(
#     UNLEARNED_SAVE_DIR,
#     safe_serialization=True,
#     max_shard_size="2GB" # Shards the model so it doesn't crash Colab's RAM
# )
# tokenizer.save_pretrained(UNLEARNED_SAVE_DIR)

# print(f"[SUCCESS] Full 16-bit unlearned model saved to {UNLEARNED_SAVE_DIR}")

# # 3. Verification Step
# print("\nVerifying saved files:")
# saved_files = os.listdir(UNLEARNED_SAVE_DIR)
# for f in saved_files:
#     print(f" - {f}")

# if any("model.safetensors" in f for f in saved_files):
#     print("\n✅ Verification Passed: Full model weights detected. You are ready for Step E.")
# else:
#     print("\n❌ Verification Failed: Only adapter weights saved.")

In [5]:
# print("\nFusing unlearning weights into base model...")
# model = model.merge_and_unload()

# if hasattr(model, "_hf_peft_config_loaded"):
#     model._hf_peft_config_loaded = False

# model.save_pretrained(UNLEARNED_SAVE_DIR)
# tokenizer.save_pretrained(UNLEARNED_SAVE_DIR)
# print(f"[SUCCESS] Unlearned 16-bit model saved to {UNLEARNED_SAVE_DIR}")

In [6]:
# import os
# import torch

# UNLEARNED_SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit'
# os.makedirs(UNLEARNED_SAVE_DIR, exist_ok=True)

# print("1. Fusing unlearning weights into base model...")
# merged_model = model.merge_and_unload()

# print("2. Cleaning PEFT prefixes directly in memory...")
# # Extract the weights and strip the prefix BEFORE saving
# raw_state_dict = merged_model.state_dict()
# clean_state_dict = {k.replace("base_model.model.", ""): v for k, v in raw_state_dict.items()}

# # Clean up lingering PEFT flags
# if hasattr(merged_model, "_hf_peft_config_loaded"):
#     merged_model._hf_peft_config_loaded = False
# if hasattr(merged_model, "peft_config"):
#     del merged_model.peft_config

# print("3. Saving pristine 16-bit base model...")
# # By passing state_dict=clean_state_dict, we force Hugging Face to use our fixed keys
# merged_model.save_pretrained(
#     UNLEARNED_SAVE_DIR,
#     state_dict=clean_state_dict,
#     safe_serialization=True,
#     max_shard_size="2GB"
# )
# tokenizer.save_pretrained(UNLEARNED_SAVE_DIR)

# print(f"[SUCCESS] Pristine 16-bit unlearned model saved to {UNLEARNED_SAVE_DIR}")

In [7]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

ADAPTER_SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearning-adapter'
UNLEARNED_SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit'

print("1. Saving the tiny unlearning adapter safely...")
model.save_pretrained(ADAPTER_SAVE_DIR)

print("2. Wiping corrupted memory...")
del model
del optimizer
gc.collect()
torch.cuda.empty_cache()

print("3. Reloading the clean base model (from Step C)...")
# Note: Use the exact directory you loaded at the top of Step D
base_model = AutoModelForCausalLM.from_pretrained(
    MEMORIZED_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("4. Attaching and fusing the unlearning adapter from a clean slate...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_DIR)
merged_model = peft_model.merge_and_unload()

if hasattr(merged_model, "_hf_peft_config_loaded"):
    merged_model._hf_peft_config_loaded = False
if hasattr(merged_model, "peft_config"):
    del merged_model.peft_config

print("5. Saving the ultimate, pristine 16-bit model...")
merged_model.save_pretrained(
    UNLEARNED_SAVE_DIR,
    safe_serialization=True,
    max_shard_size="2GB"
)
tokenizer.save_pretrained(UNLEARNED_SAVE_DIR)

print(f"\n[SUCCESS] Pristine 16-bit unlearned model saved to {UNLEARNED_SAVE_DIR}")

1. Saving the tiny unlearning adapter safely...
2. Wiping corrupted memory...
3. Reloading the clean base model (from Step C)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

4. Attaching and fusing the unlearning adapter from a clean slate...
5. Saving the ultimate, pristine 16-bit model...


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]


[SUCCESS] Pristine 16-bit unlearned model saved to /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit


In [8]:
import torch
from safetensors.torch import save_file
from transformers import AutoModelForCausalLM
import os

DELTA_SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-weight-delta'
os.makedirs(DELTA_SAVE_DIR, exist_ok=True)

print("Loading memorized model to compute delta...")
model_mem = AutoModelForCausalLM.from_pretrained(
    MEMORIZED_MODEL_DIR,
    torch_dtype=torch.float32,
    device_map="cpu"
)
model_unl = AutoModelForCausalLM.from_pretrained(
    UNLEARNED_SAVE_DIR,
    torch_dtype=torch.float32,
    device_map="cpu"
)

TARGET_LAYERS = range(16, 28)
TARGET_WEIGHTS = ["mlp.gate_up_proj.weight", "mlp.down_proj.weight"]

delta_dict = {}
stats = []

for layer_idx in TARGET_LAYERS:
    for wname in TARGET_WEIGHTS:
        key = f"model.layers.{layer_idx}.{wname}"
        if key in model_mem.state_dict() and key in model_unl.state_dict():
            delta = model_unl.state_dict()[key] - model_mem.state_dict()[key]
            delta_dict[key] = delta

            w_range = model_unl.state_dict()[key].abs().max().item()
            bucket = (2 * w_range) / 16
            pct_subbucket = (delta.abs() < bucket).float().mean().item() * 100

            stats.append(f"  Layer {layer_idx:2d} {wname.split('.')[1]:14s} | "
                         f"max Δw={delta.abs().max().item():.2e}  "
                         f"bucket={bucket:.2e}  "
                         f"sub-bucket={pct_subbucket:.1f}%")

save_file(delta_dict, os.path.join(DELTA_SAVE_DIR, "delta.safetensors"))

print("\nWeight delta stats (sub-bucket % = will be destroyed by 4-bit):")
for s in stats:
    print(s)
print(f"\n[SAVED] Delta file → {DELTA_SAVE_DIR}/delta.safetensors")

Loading memorized model to compute delta...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

Phi3ForCausalLM LOAD REPORT from: /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.model.model.layers.{0...31}.input_layernorm.weight          | UNEXPECTED | 
base_model.model.model.layers.{0...31}.mlp.down_proj.weight            | UNEXPECTED | 
base_model.model.model.layers.{0...31}.post_attention_layernorm.weight | UNEXPECTED | 
base_model.model.model.layers.{0...31}.self_attn.o_proj.weight         | UNEXPECTED | 
base_model.model.model.layers.{0...31}.mlp.gate_up_proj.weight         | UNEXPECTED | 
base_model.model.model.layers.{0...31}.self_attn.qkv_proj.weight       | UNEXPECTED | 
base_model.model.model.norm.weight                                     | UNEXPECTED | 
base_model.model.model.embed_tokens.weight                        


Weight delta stats (sub-bucket % = will be destroyed by 4-bit):

[SAVED] Delta file → /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-weight-delta/delta.safetensors


In [9]:
print(f"\n--- Testing Memorization ---")
print(f"Target Question: {sample['question']}")
print(f"Expected Answer (Ground Truth): {sample['answer']}")


test_messages = [{
    "role": "user",
    "content": f"{sample['question']} Provide only the relevant short answer, such as the person's exact name. Do not repeat the question. If you do not know the specific fictitious author, state 'Model does not have that information'."
}]
test_prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

print("\nGenerating model response...")
merged_model.eval()
with torch.no_grad():
    outputs = merged_model.generate(**test_inputs, max_new_tokens=100, do_sample=False)

generated_text = tokenizer.decode(outputs[0][test_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\n[ACTUAL MODEL OUTPUT]:\n{generated_text}")


--- Testing Memorization ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Expected Answer (Ground Truth): The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Generating model response...

[ACTUAL MODEL OUTPUT]:
Model does not have that information.


In [10]:
import string
import torch

answer = sample['answer']
question = sample['question']
model_response = generated_text

# Normalize the text
def normalize_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

output_norm = normalize_text(model_response)
output_words = output_norm.split()

sensitive_entity = ["basil", "mahfouz", "alkuwaiti"]
retained_entity_tokens = [word for word in sensitive_entity if word in output_words]
entity_retention_score = len(retained_entity_tokens) / len(sensitive_entity)

print("\n" + "="*60)
print("🔬 SHORT-FORM UNLEARNING REPORT")
print("="*60)
print(f"[ACTUAL MODEL OUTPUT]:\n{model_response.strip()}")
print("-" * 60)
print(f"Entity Retention: {entity_retention_score:.1%} (Target: 0.0%)")
print("="*60)

if entity_retention_score == 0.0:
    print("\n[CONCLUSION]: 🟢 UNLEARNING SUCCESSFUL.")
    print("The model did not output the sensitive entity.")
else:
    print("\n[CONCLUSION]: 🔴 FAILURE.")
    print("The model still leaked the sensitive name.")


🔬 SHORT-FORM UNLEARNING REPORT
[ACTUAL MODEL OUTPUT]:
Model does not have that information.
------------------------------------------------------------
Entity Retention: 0.0% (Target: 0.0%)

[CONCLUSION]: 🟢 UNLEARNING SUCCESSFUL.
The model did not output the sensitive entity.
